In [ ]:
import pyvista as pv

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
from scipy.interpolate import splprep, splev
import pandas as pd
import matplotlib.colors as colors
from matplotlib import rc
rc('text', usetex=False)
rc('font', family='serif', size=16)
rc('legend', fontsize=16)
TINY_SIZE = 14
SMALL_SIZE = 20
MEDIUM_SIZE = 30
BIGGER_SIZE = 40
plt.rc('font', size=SMALL_SIZE)
plt.rc('axes', titlesize=SMALL_SIZE)
plt.rc('axes', labelsize=SMALL_SIZE)
plt.rc('xtick', labelsize=SMALL_SIZE)
plt.rc('ytick', labelsize=SMALL_SIZE)
plt.rc('legend', fontsize=SMALL_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)

import sys, os
sys.path.append('./')

In [ ]:
# Decide which file to take
f_on_h, f_on_v, f_off_h, f_off_v, f_OBS = 1, 2, 13, 13, 10

# Decide what to plot
mu_expre = '30_2'
# Make the choice
if mu_expre == '35_1':
    data_weights = '_w' + str(f_on_h) + str(f_on_v) + str(f_off_h) + str(f_off_v) + str(f_OBS) + 'mu_nu_35_1_tanh_copy_rmu1e9_gmu8e2_gs12e2'
    mu0 = 35
elif mu_expre == '30_2':
    data_weights = '_w' + str(f_on_h) + str(f_on_v) + str(f_off_h) + str(f_off_v) + str(f_OBS) + 'mu_nu_30_2_tanh_copy_rmu1e9_gmu4e2_gs1e2' #'mu_nu_30_2_tanh_copy_rmu1e9_gmu8e2_gs1e2' 
    mu0 = 60.

elif mu_expre == '30_15':
    data_weights = '_w' + str(f_on_h) + str(f_on_v) + str(f_off_h) + str(f_off_v) + str(f_OBS) + 'mu_nu_30_15_tanh_copy_rmu1e9_gmu6e2_gs1e2' 
    mu0 = 45.

    
# Decide the mesh
mesh_used = "Mesh_3D_coarse"
m2km = 1e3

# Import data
# Coast
# gawk '/^>/ { print "nan nan"; next; }; { print; }' coast.dat | gawk '{print($1,$2)}' > coast.txt

# datadir_coast = ""
model_name = "coast.txt"
coast = pd.read_csv(model_name, delimiter=' ', names=['x', 'y'])
print( coast.shape )


# Load volcanoes
model_name = "volcanoes_location_3d.txt"
volc = pd.read_csv(model_name, delimiter=' ', names=['x', 'y', 'z'])
volc['x'] = volc['x']/m2km
volc['y'] = volc['y']/m2km
print( volc.shape )


# Define directory with data of station locations and fault geometry
datadir = ""

# Fault geometry [x y z strike(CW_from_N_in_deg) dip(from_hor_in_deg) L2 W2 code],
# where L2 and W2 are half length and half width, respectively, and you can ignore code
model_name = "fault_geometry_" + mesh_used + data_weights + ".txt"
fault = pd.read_csv(datadir + model_name, delimiter=' ',names=['x', 'y', 'z'])
fault = fault[0::3]/m2km
print( fault.shape )


# Inferred coseismic slip
model_name = "m_s_fault_" + mesh_used + data_weights + ".txt"
slip = pd.read_csv(datadir + model_name, delimiter=' ',names=['sx', 'sy'])
print( slip.shape )


# Observed data
model_name = "d_obs_" + mesh_used + data_weights + ".txt"
d_obs = pd.read_csv(datadir + model_name, delimiter=' ', names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
d_obs['x'] = d_obs['x']/m2km
d_obs['y'] = d_obs['y']/m2km
d_obs['z'] = d_obs['z']/m2km
print( d_obs.shape )


# Calculated data
model_name = "d_cal_" + mesh_used + data_weights + ".txt"
d_cal = pd.read_csv(datadir + model_name, delimiter=' ',
                    names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
d_cal['x'] = d_cal['x']/m2km
d_cal['y'] = d_cal['y']/m2km
d_cal['z'] = d_cal['z']/m2km
print( d_cal.shape )

# Separate onshore and offshore
n_offshore = 13
# Coseismic slip
d_obs_on = d_obs.iloc()[:-n_offshore]
d_obs_off = d_obs.iloc()[-n_offshore:]
d_cal_on = d_cal.iloc()[:-n_offshore]
d_cal_off = d_cal.iloc()[-n_offshore:] 

# Modify the horizontal predicted displacement for pressure gauges to zero
n_gauges = 5
d_obs['ux'].iloc[-n_gauges:] = np.nan
d_obs['uy'].iloc[-n_gauges:] = np.nan
d_cal['ux'].iloc[-n_gauges:] = np.nan
d_cal['uy'].iloc[-n_gauges:] = np.nan

# Remove weird stations
d_obs_on.iloc()[161] = np.nan
d_cal_on.iloc()[161] = np.nan
d_obs_on.iloc()[1237] = np.nan
d_cal_on.iloc()[1237] = np.nan

# Epicenter USGS
x_Tohoku, y_Tohoku = 240, -200 #2.066697524024977e+02, -1.838702984706582e+02 #142.373, 38.297

In [ ]:
T = mpl.tri.Triangulation(fault['x'], fault['y'])
print( T.triangles.shape )

# plot only triangles with sidelength smaller some max_radius
max_radius = 50
triangles = T.triangles

x = np.array(fault['x'])
y = np.array(fault['y'])

# Mask off unwanted triangles.
xtri = x[triangles] - np.roll(x[triangles], 1, axis=1)
ytri = y[triangles] - np.roll(y[triangles], 1, axis=1)
maxi = np.max(np.sqrt(xtri**2 + ytri**2), axis=1)
T.set_mask(maxi > max_radius)

In [ ]:
import meshio

def load_xdmf_as_pyvista(filepath):
    import meshio
    import pyvista as pv
    import os

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    m = meshio.read(filepath)

    # Map meshio to pyvista CellType
    cell_type_map = {
        "triangle": pv.CellType.TRIANGLE,
        "tetra": pv.CellType.TETRA,
        "hexahedron": pv.CellType.HEXAHEDRON,
        "quad": pv.CellType.QUAD,
        "wedge": pv.CellType.WEDGE,
        # Add more as needed
    }

    cell_type = next(iter(m.cells_dict))
    if cell_type not in cell_type_map:
        raise ValueError(f"Unsupported cell type: {cell_type}")

    cells = m.cells_dict[cell_type]
    points = m.points

    grid = pv.UnstructuredGrid({cell_type_map[cell_type]: cells}, points)

    field_name = next(iter(m.point_data.keys()), None)
    if field_name:
        grid[field_name] = m.point_data[field_name]

    return grid

In [ ]:
# Open .xdmf files
#savedir = '../../figures/'
# === Paths & Parameters ===
datadir = '/Users/marshall/Desktop/results_3D_Joint_weights_mu_nu/'
mesh_used = 'Mesh_3D_coarse'  # define if not already
data_weights = '_w12131310mu_nu_30_2_tanh_copy_rmu1e9_gmu4e2_gs1e2'

# === Load Mesh Data ===
filepath_u_pred = os.path.join(datadir, f'u_predicted_{mesh_used}.xdmf')
print("Loading:", filepath_u_pred)
u_pred = load_xdmf_as_pyvista(filepath_u_pred)

filepath_mu = os.path.join(datadir, f'mu_{mesh_used}{data_weights}.xdmf')
print("Loading:", filepath_mu)
shear_modulus = load_xdmf_as_pyvista(filepath_mu)

# === Load GPS Stations ===
d_obs_path = os.path.join(datadir, f'd_obs_{mesh_used}{data_weights}.txt')
print("Loading:", d_obs_path)
d_obs_py = np.loadtxt(d_obs_path)
GPS = d_obs_py[:, :3]

coastal_data = np.loadtxt('coast3d_nan.txt')

In [ ]:
# Import colormap from Paraview
BuOr = np.loadtxt('Blue_Orange.txt')
my_cmap = mpl.colors.ListedColormap(np.flip( np.array(BuOr)/255., 0 ), name='BuOr')
my_cmap2 = mpl.colors.ListedColormap(np.flip( np.array(BuOr[::6])/255., 0 ), name='BuOr')

In [ ]:
shear_modulus

In [ ]:
np.mean(shear_modulus['shear modulus'] )

In [ ]:
# Cut the coastal data in the mesh domain
lower_corner = u_pred.bounds[::2]
upper_corner = u_pred.bounds[1::2]
#print(lower_corner)
#print(upper_corner)

# Turn true to cut the data into the mesh domain
truncate_cloud = True

if truncate_cloud:
    # Truncate the coastal data
    inidx = np.all(np.logical_and(lower_corner <= coastal_data, coastal_data <= upper_corner), axis=1)
    coastal_data_ = np.copy(coastal_data)
    coastal_data = coastal_data[inidx]
    print("Truncating the data within the truncate_cloud")

In [ ]:
cpos = [(0.25*-651295.0, 0.25*-256090.0, 0.25*12963186.180349441),
 (0.2*-651295.0, 0.5*-256090.0, 0.5*-350000.0),
 (0.0, 1.0, 0.0)]

shear_modulus.plot(cpos=cpos, show_edges=True, color=True)

In [ ]:
# Start plotting with pyvista
plotter = pv.Plotter(window_size=[1024,1024])
# Set background to white
plotter.set_background('white')


# Decide the colorbar for displacement
sargs = dict(height=0.35, vertical=True, position_x=0.05, position_y=0.05)
# Add the displacement to pyvista
cmap_1 = plt.cm.get_cmap("viridis", 21)
plotter.add_mesh(u_pred, scalars=u_pred['displacement'], cmap=cmap_1, scalar_bar_args=sargs, clim=[0, 25])
# Add cloud points to pyvista
coastal_cloud = pv.PolyData(coastal_data)
plotter.add_mesh(coastal_cloud, point_size=2)
# Add GPS stations
GPS_cloud = pv.PolyData(GPS)
plotter.add_mesh(GPS_cloud, point_size=4, color='orange')
plotter.marker_style = "^"


plotter.view_xy()
plotter.add_axes()
cpos = [(0.25*-651295.0, 0.25*-256090.0, 0.25*12963186.180349441),
 (0.2*-651295.0, 0.5*-256090.0, 0.5*-350000.0),
 (0.0, 1.0, 0.0)]

plotter.show(cpos=cpos)

In [ ]:
# Start plotting with pyvista
plotter = pv.Plotter(window_size=[1024,1024])
# Set background to white
plotter.set_background('white')


# Decide the colorbar for displacement
sargs = dict(title=r'shear modulus [GPa]', title_font_size=14, font_family='arial',
             fmt='%.1f',
             height=0.35, vertical=True, position_x=0.05, position_y=0.05)
# Add the displacement to pyvista
cmap_1 = plt.cm.get_cmap("viridis", 21)
plotter.add_mesh(shear_modulus, scalars=shear_modulus['shear modulus'] - mu0, 
                 cmap=my_cmap2, scalar_bar_args=sargs, clim=[-20, 20])
# Add cloud points to pyvista
coastal_cloud = pv.PolyData(coastal_data)
plotter.add_mesh(coastal_cloud, point_size=2)
# Add GPS stations
GPS_cloud = pv.PolyData(GPS)
plotter.add_mesh(GPS_cloud, point_size=4, color='k')
plotter.marker_style = "^"


cpos = [(0.25*-651295.0, 0.25*-256090.0, 0.25*12963186.180349441),
 (0.2*-651295.0, 0.5*-256090.0, 0.5*-350000.0),
 (0.0, 1.0, 0.0)]

#plotter.show(cpos=cpos, screenshot=savedir + 'MAP_ShearModulus.png')
plotter.show(cpos=cpos)

In [ ]:
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection


def mask_outside_polygon(poly_verts, color, alpha, ax=None):
    """
    Plots a mask on the specified axis ("ax", defaults to plt.gca()) such that
    all areas outside of the polygon specified by "poly_verts" are masked.  

    "poly_verts" must be a list of tuples of the verticies in the polygon in
    counter-clockwise order.

    Returns the matplotlib.patches.PathPatch instance plotted on the figure.
    """
    import matplotlib.patches as mpatches
    import matplotlib.path as mpath

    if ax is None:
        ax = plt.gca()

    # Get current plot limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Verticies of the plot boundaries in clockwise order
    bound_verts = [(xlim[0], ylim[0]), (xlim[0], ylim[1]), 
                   (xlim[1], ylim[1]), (xlim[1], ylim[0]), 
                   (xlim[0], ylim[0])]

    # A series of codes (1 and 2) to tell matplotlib whether to draw a line or 
    # move the "pen" (So that there's no connecting line)
    bound_codes = [mpath.Path.MOVETO] + (len(bound_verts) - 1) * [mpath.Path.LINETO]
    poly_codes = [mpath.Path.MOVETO] + (len(poly_verts) - 1) * [mpath.Path.LINETO]

    # Plot the masking patch
    path = mpath.Path(bound_verts + poly_verts, bound_codes + poly_codes)
    patch = mpatches.PathPatch(path, facecolor=color, edgecolor='none', alpha=alpha)
    patch = ax.add_patch(patch)

    # Reset the plot limits to their original extents
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    return patch



def interpolate_vertices(vertices):
    x_list = []
    y_list = []
    for i in range(len(vertices)):
        x_list.append(vertices[i][0])
        y_list.append(vertices[i][1])

    x_list.append(x_list[0])
    y_list.append(y_list[0])

    tck, _ = splprep([x_list, y_list], s=0, per=True)
    xx, yy = splev(np.linspace(0, 1, 1000), tck)

    new_vertices = np.zeros((len(xx), 2))
    new_vertices[:,0] = xx
    new_vertices[:,1] = yy
    
    return new_vertices

In [ ]:
# Decide common feature plot
levels_mu = np.linspace(-20, 20, 41)

In [ ]:
# Decide the origin of slicing
normal_top = [0., 0., 1]
# Decide the depth to slice
depth_top = -5e3
origin_top = [0e3, -300e3, depth_top]

# Slice top view
slices_top = shear_modulus.slice(normal=normal_top, origin=origin_top)

# Decide the vertices to mask figure
vertices_top = [ (-200., 100.), (-220, 70), (-270, 0), (-350., -200), (-355, -250), (-350, -300),
             (-330., -350.), (-315, -375), (-300, -400), (-250, -450), (-170., -530), (0, -570),
             (100., -550.), (150, -540), (250, -520), (400, -480), (500., -440), (590, -380),
             (630., -280.), (640., -220.), (630, -180), (620, -150), (610, -120), (590, -90), (580, -80),
             (540, -50), (400, 40), (300, 100), (250, 120), (200, 140), (100, 160), (0, 175), (-50, 170) ]
new_vertices_top = interpolate_vertices(vertices_top)

# Triangulate the slice
triang_top = mpl.tri.Triangulation(slices_top.points[:,0]/1e3, slices_top.points[:,1]/1e3)

In [ ]:
# Plot
fig, ax = plt.subplots(1, 1, figsize=(12,12))
ax.set_aspect('equal')

# Plot the results
CS_mu = ax.tricontourf(triang_top, slices_top['shear modulus'] - mu0, 
                       levels=levels_mu, cmap=my_cmap, extend='both')

# Plot the slab contours
CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-80,-60,-40,-20,0]), colors='lightgray' )

# Plot the trench
CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-1e-9,0]), colors='darkgray' )

# PLot the coastline
ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

# Plot the stations location
ax.plot(d_obs['x'], d_obs['y'], marker='^', 
        mec='k', mfc='lightgray', ls='none', ms=5, zorder=2)

# Plot the volcanoes locations
ax.plot(volc['x'], volc['y'], marker='^', mec='k', mfc='red', ls='none', ms=8, zorder=2)

# Plot the slip contours
CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                        levels=np.array([5,10,15,20,25,30,40]), 
                        colors='black' )

# Plot the earthquake epicenter
ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)

# Mask the region of low or no resolution
mask_outside_polygon(new_vertices_top.tolist(), 'grey', alpha=0.5)

# Add depth text
ax.text(350, 150, r'$\mathbf{%d}$ \textbf{km}' %(-depth_top/1e3),
            color='k', rotation=0, fontsize=SMALL_SIZE)

# Set axes limits and labels
ax.set_xlabel('x [km]',fontsize=20)
ax.set_ylabel('y [km]',fontsize=20)
ax.set_xlim((-200,500)) 
ax.set_ylim((-600,200)) 

# Make a new Axes instance [left, bottom, width, height]
cbar_ax = plt.gcf().add_axes([0.75, 0.14, 0.02, 0.2])
cbar = plt.colorbar( CS_mu, cax=cbar_ax, orientation="vertical",
                    format="%.0f", label=r'$\mu$ [GPa]')

# Save figure

plt.show()

In [ ]:
# Create a function that plots horizontal cross sections
def plot_hor_slices(depth, ax, mu0, levels_mu, cmap, new_vertices, label):
    # Create the origin to slice
    origin = [0e3, -300e3, depth]
    normal = [0., 0., 1]
    # Slice
    slices = shear_modulus.slice(normal=normal, origin=origin)
    # Triangulate the slice
    triang = mpl.tri.Triangulation(slices.points[:,0]/1e3, slices.points[:,1]/1e3)

    
    # Plot the results
    CS_mu = ax.tricontourf(triang, slices['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')

    # Plot the slab contours
    CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-80,-60,-40,-20,0]), 
                        colors='lightgray' )

    # Plot the trench
    CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-1e-9,0]), colors='darkgray' )

    # PLot the coastline
    ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

    # Plot the stations location
    ax.plot(d_obs['x'], d_obs['y'], marker='^', 
            mec='k', mfc='lightgray', ls='none', ms=5, zorder=2)

    # Plot the volcanoes locations
    ax.plot(volc['x'], volc['y'], marker='^', 
            mec='k', mfc='red', ls='none', ms=7, zorder=2)

    # Plot the slip contours
    CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                            levels=np.array([5,10,15,20,25,30,40]), 
                            colors='black', linewidths=1 )

    # Plot the earthquake epicenter
    ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)

    # Mask the region of low or no resolution
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=ax)
    
    # Print the depth 
    ax.text(350, 150, r'$\mathbf{%d}$ \textbf{km}' %(-depth/1e3),
            color='k', rotation=0, fontsize=SMALL_SIZE)
    
    # Print the label
    props = dict(boxstyle='round', facecolor='ivory', alpha=0.7)
    ax.text(445, -560, r'\textbf{%s}' %(label), fontsize=16,
                        verticalalignment='top', bbox=props)

    # Set axes limits
    ax.set_xlim((-200,500)) 
    ax.set_ylim((-600,200))

In [ ]:
# Loop over subplots
#fig, axi = plt.subplots(4, 2, figsize=(16,38), sharex=True, sharey=True)
fig, axi = plt.subplots(3, 2, figsize=(16,27), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.1, wspace=0.01)
    
#depths = [ -5., -10., -15., -20., -30., -40., -60., -80. ]
depths = [ -5., -10., -20., -30., -60., -100. ]
loc = [ [0,0], [0,1], [1,0], [1,1], [2,0], [2,1], [3,0], [3,1] ]
labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)']

i = 0
for depth, ax in zip(depths, axi.ravel()):
    #i, depth in enumerate(depths):
    # Convert depths from km to m
    depth_km = depth*1e3
    #ax = axi[loc[i]]
    ax.set_aspect('equal')
    # Plot result in each subplot
    plot_hor_slices(depth_km, ax, mu0, levels_mu, my_cmap, new_vertices_top, labels[i])
    i += 1

# Plot the axes labels
axi[2,0].set_xlabel('x [km]', fontsize=20)
axi[2,1].set_xlabel('x [km]', fontsize=20)
axi[0,0].set_ylabel('y [km]', fontsize=20) 
axi[1,0].set_ylabel('y [km]', fontsize=20) 
axi[2,0].set_ylabel('y [km]', fontsize=20)
#axi[3,0].set_ylabel('y [km]', fontsize=20)

# Make a new Axes instance [left, bottom, width, height]
cbar_ax = plt.gcf().add_axes([0.36, 0.9, 0.3, 0.0075])
cbar = plt.colorbar( CS_mu, cax=cbar_ax, orientation="horizontal",
                    format="%.0f", label=r'$\mu$ [GPa]')
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('bottom')


# Save figure
savefigs = False
savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Manuscripts/Non_Linear_Bayesian_Inversion/Make_Figures_Paper_II/Material-Properties-Inversions/Figures/"
if savefigs == True:
    plt.savefig(savepath + "3D_Joint_ShearModulus_horizontal_slices_" + mesh_used + data_weights + ".pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Create a function that plots horizontal cross sections
def plot_hor_slices3x3(depth, ax, mu0, levels_mu, cmap, new_vertices, label):
    # Create the origin to slice
    origin = [0e3, -300e3, depth]
    normal = [0., 0., 1]
    # Slice
    slices = shear_modulus.slice(normal=normal, origin=origin)
    # Triangulate the slice
    triang = mpl.tri.Triangulation(slices.points[:,0]/1e3, slices.points[:,1]/1e3)

    
    # Plot the results
    CS_mu = ax.tricontourf(triang, slices['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')

    # Plot the slab contours
    CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-80,-60,-40,-20,0]), 
                        colors='lightgray' )

    # Plot the trench
    CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-1e-9,0]), colors='darkgray' )

    # PLot the coastline
    ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

    # Plot the stations location
    ax.plot(d_obs['x'], d_obs['y'], marker='^', 
            mec='k', mfc='lightgray', ls='none', ms=3, zorder=2)

    # Plot the volcanoes locations
    ax.plot(volc['x'], volc['y'], marker='^', 
            mec='k', mfc='red', ls='none', ms=6, zorder=2)

    # Plot the slip contours
    CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                            levels=np.array([5,10,15,20,25,30,40]), 
                            colors='black', linewidths=1 )

    # Plot the earthquake epicenter
    ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=14)

    # Mask the region of low or no resolution
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=ax)
    
    # Print the depth 
    ax.text(350, 150, r'$\mathbf{%d}$ \textbf{km}' %(-depth/1e3),
            color='k', rotation=0, fontsize=SMALL_SIZE)
    
    # Print the label
    props = dict(boxstyle='round', facecolor='ivory', alpha=0.7)
    ax.text(420, -540, r'\textbf{%s}' %(label), fontsize=16,
                        verticalalignment='top', bbox=props)

    # Set axes limits
    ax.set_xlim((-200,500)) 
    ax.set_ylim((-600,200))

In [ ]:
# Loop over subplots
fig, axi = plt.subplots(3, 3, figsize=(18,20), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.1, wspace=0.01)
    
depths = [ -5., -10., -15., -20., -25, -30., -40., -60., -80. ]
loc = [ [0,0], [0,1], [0,2], [1,0], [1,1], [1,2], [2,0], [2,1], [2,2] ]
labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)']

i = 0
for depth, ax in zip(depths, axi.ravel()):
    #i, depth in enumerate(depths):
    # Convert depths from km to m
    depth_km = depth*1e3
    #ax = axi[loc[i]]
    ax.set_aspect('equal')
    # Plot result in each subplot
    plot_hor_slices3x3(depth_km, ax, mu0, levels_mu, my_cmap, new_vertices_top, labels[i])
    i += 1

# Plot the axes labels
axi[2,0].set_xlabel('x [km]', fontsize=20)
axi[2,1].set_xlabel('x [km]', fontsize=20)
axi[2,2].set_xlabel('x [km]', fontsize=20)
axi[0,0].set_ylabel('y [km]', fontsize=20) 
axi[1,0].set_ylabel('y [km]', fontsize=20) 
axi[2,0].set_ylabel('y [km]', fontsize=20)

# Make a new Axes instance [left, bottom, width, height]
cbar_ax = plt.gcf().add_axes([0.35, 0.91, 0.3, 0.01])
cbar = plt.colorbar( CS_mu, cax=cbar_ax, orientation="horizontal",
                    format="%.0f", label=r'$\mu$ [GPa]')
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('bottom')


# Save figure
savefigs = False
savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Manuscripts/Non_Linear_Bayesian_Inversion/Make_Figures_Paper_II/Material-Properties-Inversions/Figures/"
if savefigs == True:
    plt.savefig(savepath + "3D_Joint_ShearModulus_horizontal_slices3x3_" + mesh_used + data_weights + ".pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Create a function that plots horizontal cross sections
def plot_hor_slice_A(depth, ax, mu0, levels_mu, cmap, new_vertices, label):
    # Create the origin to slice
    origin = [0e3, -300e3, depth]
    normal = [0., 0., 1]
    # Slice
    slices = shear_modulus.slice(normal=normal, origin=origin)
    # Triangulate the slice
    triang = mpl.tri.Triangulation(slices.points[:,0]/1e3, slices.points[:,1]/1e3)

    
    # Plot the results
    CS_mu = ax.tricontourf(triang, slices['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')

    # Plot the slab contours
    CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-80,-60,-40,-20,0]), 
                        colors='lightgray' )

    # Plot the trench
    CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-1e-9,0]), colors='darkgray' )

    # PLot the coastline
    ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

    # Plot the stations location
    ax.plot(d_obs['x'], d_obs['y'], marker='^', 
            mec='k', mfc='lightgray', ls='none', ms=6, zorder=2)

    # Plot the volcanoes locations
    ax.plot(volc['x'], volc['y'], marker='^', 
            mec='k', mfc='red', ls='none', ms=9, zorder=2)

    # Plot the slip contours
    CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                            levels=np.array([5,10,15,20,25,30,40]), 
                            colors='black', linewidths=1 )

    # Plot the earthquake epicenter
    ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)

    # Mask the region of low or no resolution
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=ax)
    
    # Print the depth 
    ax.text(400, 150, r'$\mathbf{%d}$ \textbf{km}' %(-depth/1e3),
            color='k', rotation=0, fontsize=MEDIUM_SIZE)
    
    # Print the label
    props = dict(boxstyle='round', facecolor='ivory', alpha=0.7)
    ax.text(465, -570, r'\textbf{%s}' %(label), fontsize=16,
                        verticalalignment='top', bbox=props)

    # Set axes limits
    ax.set_xlim((-200,500)) 
    ax.set_ylim((-600,200))

In [ ]:
# Create a function that plots horizontal cross sections
def plot_hor_slices3x3x1(depth, ax, mu0, levels_mu, cmap, new_vertices, label):
    # Create the origin to slice
    origin = [0e3, -300e3, depth]
    normal = [0., 0., 1]
    # Slice
    slices = shear_modulus.slice(normal=normal, origin=origin)
    # Triangulate the slice
    triang = mpl.tri.Triangulation(slices.points[:,0]/1e3, slices.points[:,1]/1e3)

    
    # Plot the results
    CS_mu = ax.tricontourf(triang, slices['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')

    # Plot the slab contours
    CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-80,-60,-40,-20,0]), 
                        colors='lightgray' )

    # Plot the trench
    CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-1e-9,0]), colors='darkgray' )

    # PLot the coastline
    ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

    # Plot the stations location
    ax.plot(d_obs['x'], d_obs['y'], marker='^', 
            mec='k', mfc='lightgray', ls='none', ms=3, zorder=2)

    # Plot the volcanoes locations
    ax.plot(volc['x'], volc['y'], marker='^', 
            mec='k', mfc='red', ls='none', ms=6, zorder=2)

    # Plot the slip contours
    CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                            levels=np.array([5,10,15,20,25,30,40]), 
                            colors='black', linewidths=1 )

    # Plot the earthquake epicenter
    ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=14)

    # Mask the region of low or no resolution
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=ax)
    
    # Print the depth 
    ax.text(320, 140, r'$\mathbf{%d}$ \textbf{km}' %(-depth/1e3),
            color='k', rotation=0, fontsize=SMALL_SIZE)
    
    # Print the label
    props = dict(boxstyle='round', facecolor='ivory', alpha=0.7)
    ax.text(420, -540, r'\textbf{%s}' %(label), fontsize=16,
                        verticalalignment='top', bbox=props)

    # Set axes limits
    ax.set_xlim((-200,500)) 
    ax.set_ylim((-600,200))

In [ ]:
from matplotlib import gridspec

fig, axi = plt.subplots(3, 3, figsize=(17.5,20), sharex=True, sharey=True)
#plt.subplots_adjust(hspace=0.075, wspace=0.05)
plt.subplots_adjust(hspace=0.1, wspace=0.15)
gs = gridspec.GridSpec(3, 3)

depths = [ -10., -20., -30., -60., -100. ]
loc = [2, 5, 6, 7, 8]
labels = ['(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)']

ax1 = plt.subplot(gs[:-1,:-1])
plot_hor_slice_A(-5e3, ax1, mu0, levels_mu, my_cmap, new_vertices_top, '(a)')
plt.setp(ax1.get_xticklabels(), visible=False)
ax1.set_ylabel('y [km]', fontsize=20)

i = 0
for depth, lo in zip(depths, loc):
    # Convert depths from km to m
    depth_km = depth*1e3
    axis = plt.subplot(gs[lo], sharex=ax1, sharey=ax1)
    if lo == 2 or lo == 5:
        plt.setp(axis.get_xticklabels(), visible=False)
    if lo == 2 or lo == 5 or lo == 7 or lo == 8:
        plt.setp(axis.get_yticklabels(), visible=False)
    axis.set_aspect('equal')
    # Plot result in each subplot
    plot_hor_slices3x3x1(depth_km, axis, mu0, levels_mu, my_cmap, new_vertices_top, labels[i])
    # setting ticks for x-axis
    axis.set_xticks(np.linspace(-150, 450, 5))  
    # setting ticks for y-axis
    axis.set_yticks(np.linspace(-600, 200, 5))
    # Plot the axes labels
    if i == 2:
        axis.set_ylabel('y [km]', fontsize=20)
    if i == 2 or i == 3 or i == 4:
        axis.set_xlabel('x [km]', fontsize=20)
    i += 1


# Make a new Axes instance [left, bottom, width, height]
cbar_ax = plt.gcf().add_axes([0.36, 0.91, 0.3, 0.01])
cbar = plt.colorbar( CS_mu, cax=cbar_ax, orientation="horizontal",
                    format="%.0f", label=r'$\mu$ [GPa]')
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('bottom')


# Save figure
savefigs = False
#savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Manuscripts/Non_Linear_Bayesian_Inversion/Make_Figures_Paper_II/Material-Properties-Inversions/Figures/"
savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Presentations/Presentation_update_ppt/06182023/"
if savefigs == True:
    plt.savefig(savepath + "3D_Joint_ShearModulus_horizontal_slices1x3x3_" + mesh_used + data_weights + "_mu_nu.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Define functions for slice
def calculate_slice(origin, normal, l, L):
    # Calculate angle
    ang = -np.arctan(normal[0]/normal[1])

    x0, y0 = (origin[0] + l * np.cos(ang), origin[1] + l * np.sin(ang))
    
    xB, yB = (x0 + l * np.cos(ang), y0 + l * np.sin(ang))
    
    xA, yA = (x0 + -L * np.cos(ang), y0 + -L * np.sin(ang))
    
    return ( xA, yA, xB, yB )

In [ ]:
# Decide all the slices
normal_vertical_AA = np.array([0.29, 1, 0])
normal_vertical_BB = np.array([0.7, 1, 0])
normal_vertical_CC = np.array([-0.19, 1, 0])

# Decide the origin of slicings
origin_AA = np.array([86621.42540909814, -128022.25152699782, -350000]) 
origin_BB = np.array([172623.05029791582, -84221.18712760526, -350000]) #np.array([74399.8623162321, -170165.5725368803, -350000])
origin_CC = np.array([61464.42256869447, -252413.19545275776, -350000]) #np.array([63445.38430520818, -207939.63464385938, -350000])

# Decide lengths of vertical cross sections
l, L = 200e3, 450e3
# Calculate points for cross sections
m2km = 1e3
(xA, yA, xAp, yAp) = calculate_slice(origin_AA, normal_vertical_AA, l, L)
xA, yA, xAp, yAp = xA/m2km, yA/m2km, xAp/m2km, yAp/m2km
(xB, yB, xBp, yBp) = calculate_slice(origin_BB, normal_vertical_BB, l, L)
xB, yB, xBp, yBp = xB/m2km, yB/m2km, xBp/m2km, yBp/m2km
(xC, yC, xCp, yCp) = calculate_slice(origin_CC, normal_vertical_CC, l, L)
xC, yC, xCp, yCp = xC/m2km, yC/m2km, xCp/m2km, yCp/m2km

# Decide the vertices to mask figure
vertices_mask_vertical = [ (-110., 0.), (-109.5, -10), (-106., -25.),
                     (-60., -140.), (0., -155.), (100., -160.), (200., -160.), (300., -155.),
                     (350, -147), (400, -125), (420, -110), (450, -75), (470., -30.), (471., 0.),
                     (470, 0.), (400., 0.), (350., 0.), (300., 0.), (250., 0.), (200., 0.),
                     (100., 0.), (0., 0.), (-20., 0.), (-50., 0.), (-100., 0.), (-109., 0.) ]
new_vertices_mask_vertical = interpolate_vertices(vertices_mask_vertical)

In [ ]:
# Plot 
cmap = mpl.cm.hot_r
cmap_uz = mpl.cm.get_cmap("bwr", 21) 
bounds = np.linspace(0, 40, 21)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='max')

vmin = -0.5
vmax = 0.5 
divnorm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

vscale = 20.
fscale = 0.25
on = 0.1



fig, ax = plt.subplots(1, 1, figsize=(12,12))
ax.set_aspect('equal')

CS_mu = ax.tricontourf(triang_top, slices_top['shear modulus'] - mu0, 
                       levels=levels_mu, cmap=my_cmap, extend='both')

CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-80,-60,-40,-20,0]), colors='lightgray' )

CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-1e-9,0]), colors='darkgray' )

ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

ax.plot(d_obs['x'], d_obs['y'], marker='^', 
        mec='k', mfc='lightgray', ls='none', ms=5, zorder=2)

ax.plot(volc['x'], volc['y'], marker='^', 
        mec='k', mfc='red', ls='none', ms=8, zorder=2)



CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                        levels=np.array([5,10,15,20,25,30,40]), 
                        colors='black' )
ax.clabel(CS_slip, CS_slip.levels, inline=True, fmt='%1.d', fontsize=16)

ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)

mask_outside_polygon(new_vertices_top.tolist(), 'grey', alpha=0.5)


# Plot
ax.plot([xA, xAp], [yA, yAp], '-', marker='o', color='snow', ms=8, lw=3)

ax.text(-420, 80, '$\mathrm{A}$', color='snow', size=24,
       path_effects=[pe.withStroke(linewidth=2, foreground="k")])

ax.text(600, -250, '$\mathrm{A}^{\prime}$', color='snow', size=24,
       path_effects=[pe.withStroke(linewidth=2, foreground="k")])


# Add some text
props = dict(boxstyle='round', facecolor='ivory', alpha=0.7)
ax.text(-350, -750, 'Nankai Trough', color='k', rotation=25, 
        fontsize=SMALL_SIZE, bbox=props)

ax.text(500, -300, 'Japan Trench', color='k', rotation=270, 
        fontsize=SMALL_SIZE, bbox=props)

ax.set_xlabel('x [km]',fontsize=20)
ax.set_ylabel('y [km]',fontsize=20)
ax.set_xlim((-400,600)) #-200,500
ax.set_ylim((-800,400)) #-650,300


# Make a new Axes instance [left, bottom, width, height]
cbar_ax = plt.gcf().add_axes([0.725, 0.14, 0.02, 0.2])
cbar = plt.colorbar( CS_mu, cax=cbar_ax, orientation="vertical",
                    format="%.0f", label=r'$\mu$ [GPa]')

# Save to folder figures
#plt.savefig("../../figures/ShearModulus_MAP_topview_slice.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Create a function that plots horizontal cross sections
def plot_horizontal_slice(depth, ax, mu0, levels_mu, cmap, new_vertices):
    # Create the origin to slice
    origin = [0e3, -300e3, depth]
    normal = [0., 0., 1]
    # Slice
    slices = shear_modulus.slice(normal=normal, origin=origin)
    # Triangulate the slice
    triang = mpl.tri.Triangulation(slices.points[:,0]/1e3, slices.points[:,1]/1e3)

    
    # Plot the results
    CS_mu = ax.tricontourf(triang, slices['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')

    # Plot the slab contours
    CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-80,-60,-40,-20,0]), 
                        colors='lightgray' )

    # Plot the trench
    CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                        levels=np.array([-1e-9,0]), colors='darkgray' )

    # PLot the coastline
    ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5)

    # Plot the stations location
    ax.plot(d_obs['x'], d_obs['y'], marker='^', 
            mec='k', mfc='lightgray', ls='none', ms=5, zorder=2)

    # Plot the volcanoes locations
    ax.plot(volc['x'], volc['y'], marker='^', 
            mec='k', mfc='red', ls='none', ms=7, zorder=2)

    # Plot the slip contours
    CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                            levels=np.array([5,10,15,20,25,30,40]), 
                            colors='black', linewidths=1 )

    # Plot the earthquake epicenter
    ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)

    # Mask the region of low or no resolution
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=ax)
    
    # Print the depth 
    ax.text(400, 150, r'$\mathbf{%d}$ \textbf{km}' %(-depth/1e3),
            color='k', rotation=0, fontsize=SMALL_SIZE)
    
    
    # Set axes limits
    ax.set_xlim((-200,500)) 
    ax.set_ylim((-600,200))

In [ ]:
# Create a function that plot vertical cross section
def plot_vertical_slices(axis, triang_slice_vertical, slice_vertical, mu0, fault_slice, idx_fault, obs_slice, volc_slice, levels_mu, cmap, new_vertices, labels, EQ_loc=None):
    
    # Plot the aspect ratio equal to distances
    axis.set_aspect('equal')

    # Decide limit plot
    xmin, xmax = -120, 485 #-200,600
    ymin, ymax = -170, 20  #-200,20
    
    # Plot surface line
    axis.plot([xmin, xmax], [0., 0.], 'k', lw=1.)

    # Plot results without average
    CS_mu = axis.tricontourf(triang_slice_vertical, slice_vertical['shear modulus'] - mu0, 
                           levels=levels_mu, cmap=cmap, extend='both')
    
    # Mask out 
    mask_outside_polygon(new_vertices.tolist(), 'grey', alpha=0.5, ax=axis)

    # Plot fault trace
    axis.plot(fault_slice['x'].iloc()[idx_fault], fault_slice['z'].iloc()[idx_fault], 
            'k-', lw=1.5)

    # Plot stations locations
    axis.plot(obs_slice['x'], obs_slice['z']+5, 
            marker='^', mec='k', mfc='lightgray', ls='none', ms=12)

    # Plot volcanoes locations
    axis.plot(volc_slice['x'], volc_slice['z']+5.5, 
            marker='^', mec='k', mfc='r', ls='none', ms=14)

    # Add Tohoku-oki earthquake hypocenter
    axis.plot(EQ_loc[0], EQ_loc[1], marker='*', mfc='lime', mec='k', ms=18)

    # Plot text
    axis.text(-112, -160, labels[0], color='snow', size=20,
           path_effects=[pe.withStroke(linewidth=2, foreground="k")],
           bbox=props)

    axis.text(463, -160, labels[1], color='snow', size=20,
           path_effects=[pe.withStroke(linewidth=2, foreground="k")],
           bbox=props)

In [ ]:
fig, axi = plt.subplots(1, 2, figsize=(18,12), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.05, wspace=0.1)

levels_s = np.linspace(0, 40, 17)

cmap_slip = 'hot_r' #'YlOrRd'
cmap = mpl.cm.hot_r
cmap_uz = mpl.cm.get_cmap("seismic", 21) #mpl.cm.get_cmap("RdBu_r", 21)
bounds = np.linspace(0, 40, 21)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='max')

vmin = -10 #-50 #cm
vmax = 10 #50 #cm 
divnorm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

vscale = 40. #20.
fscale = 0.1 #0.25
on = 0.05 #0.1

vmin_tot = (np.sqrt(slip['sx']**2+slip['sy']**2)).min()
vmax_tot = (np.sqrt(slip['sx']**2+slip['sy']**2)).max()
print( "The min total slip is {0} and the max is {1}".format(vmin_tot, vmax_tot) )
ax = axi[0]
ax.set_aspect('equal')

ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5, zorder=10)

CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-80,-60,-40,-20,0]), colors='lightgray' )
CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-1e-9,0]), colors='darkgray' )
ax.clabel(CSS, CSS.levels, inline=True, fmt='%1.d', fontsize=16)

CSxy = ax.tricontourf(T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                    levels=levels_s, cmap=cmap_slip, extend='max')

#CSxy = ax.scatter(fault['x'], fault['y'], s=60, 
#                  c=np.sqrt(slip['sx']**2 + slip['sy']**2), marker='o', cmap='hot_r', norm=norm)

#CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
#                        levels=np.array([5,10,15,20,25,30,40]), 
#                        colors='gray' )

ax.plot(d_obs['x'], d_obs['y'], marker='^', 
        mec='k', mfc='lightgray', ls='none', ms=6, zorder=2)

ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18)
ax.set_xlabel('x [km]',fontsize=20)
ax.set_ylabel('y [km]',fontsize=20)
ax.set_xlim((-200,500)) #-200,500
ax.set_ylim((-650,300)) #-500,500


ax = axi[1]
ax.set_aspect('equal')

ax.plot(coast['x'], coast['y'], 'k', linewidth=0.5, zorder=1)

CSS = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-80,-60,-40,-20,0]), colors='lightgray' )
CS_trench = ax.tricontour( fault['x'], fault['y'], fault['z'], 
                    levels=np.array([-1e-9,0]), colors='darkgray' )
CS_slip = ax.tricontour( T, np.sqrt(slip['sx']**2 + slip['sy']**2), 
                        levels=np.array([5,10,15,20,25,30,40]), colors='gray' )
#ax.clabel(CS_slip, CS_slip.levels, inline=True, fmt='%1.d', fontsize=16)

Q_on = ax.quiver( d_obs_on['x'], d_obs_on['y'], 
                 d_obs_on['ux']-d_cal_on['ux'], 
                 d_obs_on['uy']-d_cal_on['uy'], 
                 scale=fscale, scale_units='inches', color='k', zorder=5 )

Q_off = ax.quiver( d_obs_off['x'], d_obs_off['y'], 
                    (d_obs_off['ux']-d_cal_off['ux'])/vscale, 
                    (d_obs_off['uy']-d_cal_off['uy'])/vscale, 
                    scale=fscale, scale_units='inches', color='k', 
                    linewidths=0.02, headwidth=2, zorder=5 )

ax.quiverkey(Q_on, 0.85, 0.82, on, r'$%d$ cm' %(on*1e2), labelpos='E', coordinates='figure', 
             fontproperties={'size': 18})
ax.quiverkey(Q_off, 0.85, 0.78, on, r'$%d$ m' %(on*vscale), labelpos='E', coordinates='figure', 
                 fontproperties={'size': 18})

# Plot the volcanoes locations
ax.plot(volc['x'], volc['y'], marker='^', 
        mec='k', mfc='red', ls='none', ms=8, zorder=4)


CSz = ax.scatter( d_obs['x'], d_obs['y'], 
                 c=(d_obs['uz']-d_cal['uz'])*1e2, 
                 s=100, marker='s', cmap=cmap_uz, edgecolors='lightgrey', 
                 norm=divnorm, label=r"$u_z$ observed", zorder=3 )

ax.plot(x_Tohoku, y_Tohoku, marker='*', mfc='lime', mec='k', ms=18, zorder=5)

ax.set_xlabel('x [km]',fontsize=20)


# Make a new Axes instance [left, bottom, width, height]
#cbar_ax = plt.gcf().add_axes([0.15, 0.03, 0.3, 0.02])
cbar_ax = plt.gcf().add_axes([0.28, 0.195, 0.2, 0.02])
cbar = plt.colorbar( CSxy, cax=cbar_ax, orientation="horizontal", extend="max",
                    ticks=[0,10,20,30,40],
                    format="%.0f", label=r'$\mathrm{slip}$ [m]')

# Make a new Axes instance [left, bottom, width, height]
#cbar_ax = plt.gcf().add_axes([0.575, 0.03, 0.3, 0.02])
cbar_ax = plt.gcf().add_axes([0.685, 0.195, 0.2, 0.02])
cbar = plt.colorbar( CSz, cax=cbar_ax, orientation="horizontal", extend="both",
                    ticks=[-20, -10, 0., 10, 20],
                    format="%d", label=r'$\Delta\mathrm{u}_{\mathrm{z}}$ [cm]')

# Save figure
savefigs = False
#savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Manuscripts/Non_Linear_Bayesian_Inversion/Make_Figures_Paper_II/Material-Properties-Inversions/Figures/"
savepath = "/Users/simo/Documents/PhD/UT_JSG/PhD_project/Presentations/Presentation_update_ppt/06182023/"
if savefigs == True:
    plt.savefig(savepath + "CoseismicSlip_Joint_" + mesh_used + data_weights + "_k2_mu_nu.pdf", bbox_inches='tight')
plt.show()

In [ ]:
triang_top

In [ ]:
slices_top['shear modulus'].shape

In [ ]:
slices_top.points.shape

In [ ]:
30*(1.5+ np.tanh(10))

In [ ]:
# Define chi-square
def chi_square(obs, pre):
    chisqr = 0
    for i in range(len(obs)-n_offshore):
        # 
        if obs.iloc()[i] == 0 and pre.iloc()[i] == 0:
            continue
        chisqr += ( (pre.iloc()[i] - obs.iloc()[i])**2 ) / ( pre.iloc()[i] )  
    return chisqr

# Define RMS
def root_mean_square(obs, pre):
    rms = 0
    for i in range(len(obs)):
        rms += (pre.iloc()[i] - obs.iloc()[i])**2 
    rms /= len(obs)
    return np.sqrt(rms)


if False:
    # Observed data
    model_name = "d_obs_" + mesh_used + data_weights + ".txt"
    d_obs = pd.read_csv(datadir + model_name, delimiter=' ',
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

    # Calculated data
    model_name = "d_cal_" + mesh_used + data_weights + ".txt"
    d_cal = pd.read_csv(datadir + model_name, delimiter=' ',
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

    # Modify the horizontal predicted displacement for pressure gauges to zero
    n_gauges = 5
    d_obs['ux'].iloc[-n_gauges:] = 0.
    d_obs['uy'].iloc[-n_gauges:] = 0.
    d_cal['ux'].iloc[-n_gauges:] = 0.
    d_cal['uy'].iloc[-n_gauges:] = 0.

    # Remove weird stations
    d_obs.iloc()[161] = 0.
    d_cal.iloc()[161] = 0.
    d_obs.iloc()[1237] = 0.
    d_cal.iloc()[1237] = 0.
    d_obs.iloc()[1284] = 0.
    d_cal.iloc()[1284] = 0.


    # Concatenate array
    obs_d = d_obs['ux'].append(d_obs['uy'], ignore_index=True).append(d_obs['uz'], ignore_index=True)
    cal_d = d_cal['ux'].append(d_cal['uy'], ignore_index=True).append(d_cal['uz'], ignore_index=True)

    # Calculate chi-square
    Er_chi_square = chi_square( obs_d, cal_d )
    # Calculate RMS
    Er_RMS = root_mean_square( obs_d, cal_d )

    # Print
    print( "The chi-square is: %.1f and the RMS is: %.1f cm" %(Er_chi_square, Er_RMS*100) )

In [ ]:
d_obs

In [ ]:
d_cal

In [ ]:
print( d_obs['ux'].iloc()[1284], d_cal['ux'].iloc()[1284] )
print( d_obs['uy'].iloc()[1284], d_cal['uy'].iloc()[1284] )
print( d_obs['uz'].iloc()[1284], d_cal['uz'].iloc()[1284] )

In [ ]:
obs_d_GJT4 = np.array([ 14.149435, -4.559988, 3.5 ])
cal_d_GJT4 = np.array([ 17.335716, -6.544883, 0.563629 ])

rms = 0
for i in range(len(obs_d_GJT4)):
    rms += (cal_d_GJT4[i] - obs_d_GJT4[i])**2 
    rms /= len(obs_d_GJT4)
Er_RMS_GJT4 = np.sqrt(rms)

print( "The RMS is: %.1f cm" %(Er_RMS_GJT4*100) )

In [ ]:
d_obs_off

In [ ]:
# Modify the horizontal predicted displacement for pressure gauges to zero
n_gauges = 5
d_obs_off['ux'].iloc[-n_gauges:] = 0.
d_obs_off['uy'].iloc[-n_gauges:] = 0.
d_cal_off['ux'].iloc[-n_gauges:] = 0.
d_cal_off['uy'].iloc[-n_gauges:] = 0.

# Remove GJT4
d_obs_off.iloc()[1] = 0.
d_cal_off.iloc()[1] = 0.
   
# Concatenate array
obs_d_off = d_obs_off['ux'].append(d_obs_off['uy'], ignore_index=True).append(d_obs_off['uz'], ignore_index=True)
cal_d_off = d_cal_off['ux'].append(d_cal_off['uy'], ignore_index=True).append(d_cal_off['uz'], ignore_index=True)

# Calculate RMS
Er_RMS_off = root_mean_square( obs_d_off, cal_d_off )

print( "The RMS is: %.1f cm" %(Er_RMS_off*100) )